In [5]:
#imports
import sqlite3
import pandas as pd
from tqdm import tqdm
import csv

In [2]:
# database file path
DB_FILE = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_gpt_labelled.db"

GPT_TABLE = f"gpt_labelled_clean"
MOD_TABLE = GPT_TABLE+"_with_eki_tag"

TAG_COL = "eki_tag"

In [13]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

query = f"SELECT * FROM {GPT_TABLE}"
df = pd.read_sql(query, conn)

conn.close()

In [24]:
df["ekilex_tag"] = df["ekilex_tag"].fillna("")
df["tag"] = df["tag"].fillna("")
df["tag2"] = df["tag2"].fillna("")

In [15]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,T,L1,L2,L,S,tag,initial_cat,id,gpt_tags,tag2
0,37,51,8,kutsuma,,adit,elu,ellu,"Pole välistatud , et me pundi kunagi ellu kuts...",None,...,no,yes,no,yes,no,L,elt_n80,164368,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
1,54,85,9,kirjutama,,all,bänd,bändidele,Olen viimasel ajal juhutöödest loobunud ja kir...,|A|AE|AL|AS|AT|AET|ALT|AST|,...,no,no,no,no,no,A,a_n80,485607,|A|AE|AL|AS|AT|AET|ALT|AST|,A
2,81,131,8,tiirutama,,ad,tuur,tuuridel,Saime tuhandeid kirju ja meie ümber tiirutasid...,None,...,no,no,no,no,no,E,elt_n80,131533,|E|AE|EL|ET|AET|ELT|,ELT
3,84,142,14,helistama,,el,hommik,hommikust,"Mõned tüdrukud muutusid lausa tüütuks , uurisi...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,yes,no,no,no,no,T,elt_n80,328034,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT
4,92,157,2,täitma,,ad,aasta,aastal,"Sel aastal täitsime oma missiooni , mille eesm...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,yes,no,no,no,no,T,elt_n80,140113,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432097,21413464,30073669,6,kaduma,,el,aur,aurudest,+Lenka: mul juba aurudest kadus lendab,None,...,no,yes,no,yes,no,L,elt_n80,202877,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
432098,21414898,30074249,7,saama,valmis,ad,oma,omal,Rex: ma sain isegi jutuka omal valmis,None,...,no,no,no,no,no,None,elt_n80,362774,,
432099,21415454,30074507,6,laskma,,adit,datanet,datanetti,+Knight17_away: metscat lase datanetti üless :),None,...,no,yes,no,yes,no,L,elt_n80,339718,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT
432100,21415689,30074650,4,saama,,ill,kasiioo,kasiioosse,Rex: nooremad kasiioosse ei saa,None,...,no,yes,no,yes,no,L,elt_n80,288450,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT


### Create new eki_tag based on ekilex_tag

- SÜNDMUS/EVENT = EKILEX event
- AEG/TIME = EKILEX time + TIMEX
- KOHT/LOC = EKILEX location + NER LOC + EKILEX organisation sisekohakäänetes + NER ORG sisekohakäänetes
- KOHTSÜNDMUS/LOCEVENT = EKILEX location NER LOC + EKILEX organisation sisekohakäänetes + NER ORG sisekohakäänetes + EKILEX event
- KOHTSÜNDMUSAEG/LOCEVENTTIME = KOHT + SÜNDMUS + AEG
- VALDAJA/ALIVE = EKILEX alive + NER PER + EKILEX organisation väliskohakäänetes + NER ORG väliskohakäänetes 
- SEISUND/STATE = EKILEX state

- EVENT - E
- TIME - T
- LOC - L
- LOCEVENT - EL
- LOCEVENTTIME - ELT
- ALIVE - A
- STATE - S

In [18]:
mapping = {"alive":"A", "PER": "A", "location":"L", "LOC":"L", "time":"T", "event":"E", "state":"S"}


def get_tags(row):
    ekilex_tag = row["ekilex_tag"]
    morph_case = row["morph_case"]
    if ekilex_tag == "":
        return ""
    
    if ekilex_tag == "organisation":
        # ekilex org sisekohakäänetes on LOC (adit, in, ill, el)
        if morph_case in ["adit", "in", "ill", "el"]:
            ekilex_tag = "location"
        # EKILEX organisation väliskohakäänetes on ALIVE (ad, all, abl)
        elif morph_case in ["ad", "all", "abl"]:
            ekilex_tag = "alive"

    return mapping[ekilex_tag]

In [19]:
df[TAG_COL] = df.apply(get_tags, axis=1)

In [22]:
df.columns

Index(['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound',
       'morph_case', 'lemma', 'form', 'sentence', 'tags', 'timex_tag',
       'ekilex_tag', 'ner_tag', 'A', 'E', 'T', 'L1', 'L2', 'L', 'S', 'tag',
       'initial_cat', 'id', 'gpt_tags', 'tag2', 'eki_tag'],
      dtype='object')

In [25]:
df[['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound',
       'morph_case', 'lemma', 'form', 'sentence', 'tags',
       'ekilex_tag', 'tag',
       'initial_cat', 'id', 'gpt_tags', 'tag2', 'eki_tag']]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,ekilex_tag,tag,initial_cat,id,gpt_tags,tag2,eki_tag
0,37,51,8,kutsuma,,adit,elu,ellu,"Pole välistatud , et me pundi kunagi ellu kuts...",None,,L,elt_n80,164368,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT,
1,54,85,9,kirjutama,,all,bänd,bändidele,Olen viimasel ajal juhutöödest loobunud ja kir...,|A|AE|AL|AS|AT|AET|ALT|AST|,alive,A,a_n80,485607,|A|AE|AL|AS|AT|AET|ALT|AST|,A,A
2,81,131,8,tiirutama,,ad,tuur,tuuridel,Saime tuhandeid kirju ja meie ümber tiirutasid...,None,,E,elt_n80,131533,|E|AE|EL|ET|AET|ELT|,ELT,
3,84,142,14,helistama,,el,hommik,hommikust,"Mõned tüdrukud muutusid lausa tüütuks , uurisi...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,,T,elt_n80,328034,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT,
4,92,157,2,täitma,,ad,aasta,aastal,"Sel aastal täitsime oma missiooni , mille eesm...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,,T,elt_n80,140113,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432097,21413464,30073669,6,kaduma,,el,aur,aurudest,+Lenka: mul juba aurudest kadus lendab,None,,L,elt_n80,202877,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT,
432098,21414898,30074249,7,saama,valmis,ad,oma,omal,Rex: ma sain isegi jutuka omal valmis,None,,,elt_n80,362774,,,
432099,21415454,30074507,6,laskma,,adit,datanet,datanetti,+Knight17_away: metscat lase datanetti üless :),None,,L,elt_n80,339718,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT,
432100,21415689,30074650,4,saama,,ill,kasiioo,kasiioosse,Rex: nooremad kasiioosse ei saa,None,,L,elt_n80,288450,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT,


In [28]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

df.to_sql(MOD_TABLE, conn, if_exists="replace", index=False)

conn.close()

## Lemma statistics (ekilex tags vs gpt tags for every lemma)

In [29]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

query = f"SELECT * FROM {MOD_TABLE}"
d = pd.read_sql(query, conn)

conn.close()

In [30]:
d

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,L1,L2,L,S,tag,initial_cat,id,gpt_tags,tag2,eki_tag
0,37,51,8,kutsuma,,adit,elu,ellu,"Pole välistatud , et me pundi kunagi ellu kuts...",None,...,yes,no,yes,no,L,elt_n80,164368,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT,
1,54,85,9,kirjutama,,all,bänd,bändidele,Olen viimasel ajal juhutöödest loobunud ja kir...,|A|AE|AL|AS|AT|AET|ALT|AST|,...,no,no,no,no,A,a_n80,485607,|A|AE|AL|AS|AT|AET|ALT|AST|,A,A
2,81,131,8,tiirutama,,ad,tuur,tuuridel,Saime tuhandeid kirju ja meie ümber tiirutasid...,None,...,no,no,no,no,E,elt_n80,131533,|E|AE|EL|ET|AET|ELT|,ELT,
3,84,142,14,helistama,,el,hommik,hommikust,"Mõned tüdrukud muutusid lausa tüütuks , uurisi...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,no,no,no,no,T,elt_n80,328034,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT,
4,92,157,2,täitma,,ad,aasta,aastal,"Sel aastal täitsime oma missiooni , mille eesm...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,...,no,no,no,no,T,elt_n80,140113,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,ELT,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432097,21413464,30073669,6,kaduma,,el,aur,aurudest,+Lenka: mul juba aurudest kadus lendab,None,...,yes,no,yes,no,L,elt_n80,202877,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT,
432098,21414898,30074249,7,saama,valmis,ad,oma,omal,Rex: ma sain isegi jutuka omal valmis,None,...,no,no,no,no,,elt_n80,362774,,,
432099,21415454,30074507,6,laskma,,adit,datanet,datanetti,+Knight17_away: metscat lase datanetti üless :),None,...,yes,no,yes,no,L,elt_n80,339718,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT,
432100,21415689,30074650,4,saama,,ill,kasiioo,kasiioosse,Rex: nooremad kasiioosse ei saa,None,...,yes,no,yes,no,L,elt_n80,288450,|L|AL|EL|LS|LT|ALT|ELT|LST|,ELT,


In [43]:
tags = ["E", "L", "T", "A", "S", ""]

def build_percent_table(tag_col, prefix, lemma_totals):
    # Count occurrences of each tag per lemma
    counts = (
        df.groupby(["lemma", tag_col], dropna=False)
          .size()
          .unstack(fill_value=0)
    )

    # Ensure all expected tag columns exist
    counts = counts.reindex(columns=tags, fill_value=0)

    # Convert counts to percentages
    pct = counts.div(lemma_totals, axis=0).mul(100)

    # Rename columns
    pct = pct.rename(columns={
        "E": f"{prefix}_E",
        "L": f"{prefix}_L",
        "T": f"{prefix}_T",
        "A": f"{prefix}_A",
        "S": f"{prefix}_S",
        "":  f"{prefix}__",
    })

    # ELT = E + L + T percentages
    pct[f"{prefix}_ELT"] = (
        pct[f"{prefix}_E"]
        + pct[f"{prefix}_L"]
        + pct[f"{prefix}_T"]
    )

    return pct



def calculate_tag_percentages(df):
    # Total occurrences per lemma
    lemma_totals = df.groupby("lemma", dropna=False).size()

    eki_pct = build_percent_table("eki_tag", "eki", lemma_totals)
    gpt_pct = build_percent_table("tag", "gpt", lemma_totals)

    # Combine results
    result = (
        pd.concat([eki_pct, gpt_pct], axis=1)
        .reset_index()
    )

    # Desired column order
    result = result[
        [
            "lemma",
            "eki_E", "eki_L", "eki_T", "eki_ELT", "eki_A", "eki_S", "eki__",
            "gpt_E", "gpt_L", "gpt_T", "gpt_ELT", "gpt_A", "gpt_S", "gpt__",
        ]
    ]
    
    pct_cols = result.columns.drop("lemma")
    result[pct_cols] = result[pct_cols].round(2)

    return result




In [44]:

result_df = calculate_tag_percentages(d)


In [45]:
result_df

,lemma,eki_E,eki_L,eki_T,eki_ELT,eki_A,eki_S,eki__,gpt_E,gpt_L,gpt_T,gpt_ELT,gpt_A,gpt_S,gpt__
0,$,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
1,%,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
2,%-põhimõte,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
3,%-see,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
4,%line,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80548,žurnaal,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,100.00,0.00,100.00,0.00,0.00,0.00
80549,žurnalist,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.00,0.00,0.00,100.00,0.00,0.00
80550,žürii,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,54.55,0.00,54.55,27.27,0.00,18.18
80551,ω-3-rasvhape,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00


In [53]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

result_df.to_sql("gpt_labelled_clean_lemma_statistics", conn, if_exists="replace", index=False)

conn.close()

In [3]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

query = f"SELECT * FROM gpt_labelled_clean_lemma_statistics"
df = pd.read_sql(query, conn)

conn.close()

In [4]:
df

,lemma,eki_E,eki_L,eki_T,eki_ELT,eki_A,eki_S,eki__,gpt_E,gpt_L,gpt_T,gpt_ELT,gpt_A,gpt_S,gpt__
0,$,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
1,%,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
2,%-põhimõte,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
3,%-see,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
4,%line,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80548,žurnaal,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,100.00,0.00,100.00,0.00,0.00,0.00
80549,žurnalist,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.00,0.00,0.00,100.00,0.00,0.00
80550,žürii,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,54.55,0.00,54.55,27.27,0.00,18.18
80551,ω-3-rasvhape,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.00,0.00,0.00,0.00,0.00,100.00


In [6]:
df.to_csv("../results/gpt_labelled_clean_lemma_statistics.csv", encoding="utf-8", index = False, 
          sep=",", quoting=csv.QUOTE_MINIMAL)